In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("online_retail_II.csv")
df.head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


In [3]:
df.columns

Index(['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'Price', 'Customer ID', 'Country'],
      dtype='str')

In [4]:
df.dtypes

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

In [6]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

In [7]:
summary = (
    pd.DataFrame({
        'dtype': df.dtypes,
        'non_null': df.notna().sum(),
        'nulls': df.isna().sum(),
        'null_%': (df.isna().mean() * 100).round(2),
        'unique': df.nunique()
    })
    .sort_values('null_%', ascending=False)
)

print('Column-Level Data Quality Summary')
display(summary)

Column-Level Data Quality Summary


,dtype,non_null,nulls,null_%,unique
Customer ID,float64,824364,243007,22.77,5942
Description,str,1062989,4382,0.41,5698
StockCode,str,1067371,0,0.00,5305
Invoice,str,1067371,0,0.00,53628
Quantity,int64,1067371,0,0.00,1057
InvoiceDate,datetime64[us],1067371,0,0.00,47635
Price,float64,1067371,0,0.00,2807
Country,str,1067371,0,0.00,43


In [8]:
duplicate_rows = df.duplicated().sum()
quality_snapshot = pd.DataFrame({
    'Metric': [
        'Total Rows',
        'Duplicate Rows',
        'Duplicate Row %',
        'Rows with null Description %',
        'Rows with null Customer ID %',
        'Rows with invalid InvoiceDate %',
        'Rows with non-positive Price %',
        'Rows with negative Quantity %'
    ],
    'Value': [
        len(df),
        duplicate_rows,
        round(100 * duplicate_rows / len(df), 2),
        round(100 * df['Description'].isna().mean(), 2),
        round(100 * df['Customer ID'].isna().mean(), 2),
        round(100 * df['InvoiceDate'].isna().mean(), 2),
        round(100 * (df['Price'] <= 0).mean(), 2),
        round(100 * (df['Quantity'] < 0).mean(), 2)
    ]
})

print('Dataset-Level Data Quality Snapshot')
display(quality_snapshot)

Dataset-Level Data Quality Snapshot


,Metric,Value
0,Total Rows,1067371.00
1,Duplicate Rows,34335.00
2,Duplicate Row %,3.22
3,Rows with null Description %,0.41
4,Rows with null Customer ID %,22.77
5,Rows with invalid InvoiceDate %,0.00
6,Rows with non-positive Price %,0.58
7,Rows with negative Quantity %,2.15


In [9]:
df = df.dropna(subset=['Description']).copy()

# Remove duplicated rows
df = df.drop_duplicates()

# Standardize base fields used across sections
df['Invoice'] = df['Invoice'].astype(str).str.strip()
df['Is_Cancellation'] = df['Invoice'].str.startswith('C') | (df['Quantity'] < 0)
df['LineRevenue'] = df['Quantity'] * df['Price']

# Customer-level dataset (only known customers)
df_customer = df.dropna(subset=['Customer ID']).copy()
df_customer['Customer ID'] = df_customer['Customer ID'].astype('Int64')

# Sales-level dataset (keep all rows, fill guest)
df_sales = df.copy()
df_sales['Customer ID'] = df_sales['Customer ID'].astype('string').fillna('Guest')


print(f"df_sales shape   : {df_sales.shape}")
print(f"df_customer shape: {df_customer.shape}")
print(f"Cancellation rows: {df_sales['Is_Cancellation'].sum():,} ({100 * df_sales['Is_Cancellation'].mean():.2f}%)")

df_sales shape   : (1028761, 10)
df_customer shape: (797885, 10)
Cancellation rows: 19,864 (1.93%)


In [17]:
monthly_revenue = df_sales[~df_sales['Is_Cancellation']].groupby(df_sales['InvoiceDate'].dt.to_period('M'))['LineRevenue'].sum().reset_index()
monthly_revenue.columns = ['Month', 'Revenue']
display(monthly_revenue)

,Month,Revenue
0,2009-12,822483.950
1,2010-01,651155.112
2,2010-02,551504.726
3,2010-03,830915.261
4,2010-04,625280.892
5,2010-05,657705.500
6,2010-06,749537.310
7,2010-07,604778.480
8,2010-08,695251.910
9,2010-09,921696.991


In [ ]:
import matplotlib.pyplot as plt

# Plot monthly revenue
monthly_revenue.plot(x='Month', y='Revenue', kind='line', figsize=(10, 6))
plt.title('Monthly Revenue')
plt.xlabel('Month')
plt.ylabel('Revenue')
plt.xticks(rotation=45)
plt.show()